# Manatuabon qLoRA Training (Colab)

This notebook is for a **Manatuabon-specific adapter**, not the Kaggle reasoning challenge workflow.

## Goal
Train a narrow qLoRA adapter that improves:
- structured_ingest_v1 output fidelity
- council review JSON quality
- evidence request / reflection output discipline
- conservative uncertainty and refusal behavior

## Non-Goal
Do **not** train on private chain-of-thought or generic puzzle reasoning traces.

In [ ]:
# 1. Install dependencies
!pip install -q unsloth datasets trl transformers accelerate bitsandbytes

In [ ]:
# 2. Load base model + tokenizer
from unsloth import FastLanguageModel

MODEL_NAME = 'unsloth/Nemotron-3-Nano-30B-A3B'
MAX_SEQ_LENGTH = 4096

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=MAX_SEQ_LENGTH,
    dtype=None,
    load_in_4bit=True,
)

print(f'Model loaded: {MODEL_NAME}')
print(f'Tokenizer size: {len(tokenizer)}')

In [ ]:
# 3. Attach LoRA adapter
model = FastLanguageModel.get_peft_model(
    model,
    r=32,
    lora_alpha=64,
    lora_dropout=0,
    target_modules=[
        'q_proj', 'k_proj', 'v_proj', 'o_proj',
        'gate_proj', 'up_proj', 'down_proj',
    ],
    bias='none',
    use_gradient_checkpointing='unsloth',
    random_state=42,
)

model.print_trainable_parameters()

## JSONL Schema

Each training record should look like this:

```json
{
  "record_id": "judge_review_exoplanet_001",
  "task_type": "judge_review",
  "source": "curated_manatuabon",
  "domain_tags": ["exoplanets"],
  "instruction": "Return only valid JSON matching the Manatuabon review contract.",
  "input": { ... },
  "output": { ... }
}
```

Required top-level fields:
- `record_id`
- `task_type`
- `source`
- `domain_tags`
- `instruction`
- `input`
- `output`

Supported task types in this starter corpus:
- `structured_ingest`
- `skeptic_review`
- `archivist_review`
- `judge_review`
- `reflection_review`
- `evidence_request`

This dataset is deliberately mixed so the adapter learns both bundle formatting and governance behavior.

In [ ]:
# 4. Load and validate the Manatuabon SFT dataset
import json
import os
from datasets import load_dataset

DATASET_PATH = 'manatuabon_sft_examples.jsonl'  # replace with your curated dataset upload
REQUIRED_FIELDS = {
    'record_id',
    'task_type',
    'source',
    'domain_tags',
    'instruction',
    'input',
    'output',
}
ALLOWED_TASK_TYPES = {
    'structured_ingest',
    'skeptic_review',
    'archivist_review',
    'judge_review',
    'reflection_review',
    'evidence_request',
}

if not os.path.exists(DATASET_PATH):
    raise FileNotFoundError(f'Upload {DATASET_PATH} to the Colab workspace first.')

dataset = load_dataset('json', data_files=DATASET_PATH, split='train')

def validate_record(sample):
    missing = REQUIRED_FIELDS.difference(sample.keys())
    if missing:
        raise ValueError(f"Missing required fields: {sorted(missing)} in {sample.get('record_id')}")
    if sample['task_type'] not in ALLOWED_TASK_TYPES:
        raise ValueError(f"Unsupported task_type {sample['task_type']!r} in {sample.get('record_id')}")
    if not isinstance(sample['domain_tags'], list) or not sample['domain_tags']:
        raise ValueError(f"domain_tags must be a non-empty list in {sample.get('record_id')}")
    if not isinstance(sample['instruction'], str) or not sample['instruction'].strip():
        raise ValueError(f"instruction must be a non-empty string in {sample.get('record_id')}")
    if not isinstance(sample['input'], dict):
        raise ValueError(f"input must be an object in {sample.get('record_id')}")
    if not isinstance(sample['output'], dict):
        raise ValueError(f"output must be an object in {sample.get('record_id')}")
    return sample

dataset = dataset.map(validate_record)
print(dataset)
print('Validated records:', len(dataset))
print('Task types:', sorted(set(dataset['task_type'])))

In [ ]:
# 5. Convert records into SFT text and keep a raw eval slice for contract checks
SYSTEM_PROMPT = (
    'System: You are the Manatuabon observatory assistant. '
    'You must be conservative, evidence-aware, and return only valid JSON that matches the requested contract.'
)

def format_prompt(sample):
    task_type = sample['task_type']
    instruction = sample['instruction']
    input_json = json.dumps(sample['input'], ensure_ascii=False, indent=2, sort_keys=True)
    return (
        f'{SYSTEM_PROMPT}\n\n'
        f'Task Type: {task_type}\n'
        f'Instruction: {instruction}\n\n'
        'Input JSON:\n'
        f'{input_json}\n\n'
        'Assistant JSON:\n'
    )

def format_training_text(sample):
    output_json = json.dumps(sample['output'], ensure_ascii=False, indent=2, sort_keys=True)
    return {'text': f"{format_prompt(sample)}{output_json}{tokenizer.eos_token}"}

test_rows = max(1, len(dataset) // 8)
raw_split = dataset.train_test_split(test_size=test_rows, seed=42)
raw_train_data = raw_split['train']
raw_eval_data = raw_split['test']

train_data = raw_train_data.map(format_training_text, remove_columns=raw_train_data.column_names)
eval_data = raw_eval_data.map(format_training_text, remove_columns=raw_eval_data.column_names)

print(f'Train rows: {len(train_data)}')
print(f'Eval rows: {len(eval_data)}')
print(raw_eval_data[0]['record_id'])
print(train_data[0]['text'][:1200])

In [ ]:
# 6. Configure trainer
from trl import SFTConfig, SFTTrainer
from transformers import EarlyStoppingCallback

OUTPUT_DIR = './manatuabon_nemotron_qlora'

trainer = SFTTrainer(
    model=model,
    processing_class=tokenizer,
    train_dataset=train_data,
    eval_dataset=eval_data,
    args=SFTConfig(
        output_dir=OUTPUT_DIR,
        dataset_text_field='text',
        per_device_train_batch_size=1,
        gradient_accumulation_steps=4,
        warmup_steps=10,
        max_steps=200,
        learning_rate=5e-5,
        bf16=True,
        logging_steps=5,
        eval_strategy='steps',
        eval_steps=20,
        save_strategy='steps',
        save_steps=20,
        load_best_model_at_end=True,
        metric_for_best_model='eval_loss',
        optim='adamw_8bit',
        seed=42,
    ),
    callbacks=[EarlyStoppingCallback(early_stopping_patience=3)],
)

print('Trainer ready.')

In [ ]:
# 7. Train
trainer_stats = trainer.train()
print(f'Training complete. Steps: {trainer_stats.global_step}')

In [ ]:
# 8. Save adapter
model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print(f'Saved adapter to {OUTPUT_DIR}')

In [ ]:
# 9. Post-training contract check on held-out examples
from pprint import pprint

FastLanguageModel.for_inference(model)

REQUIRED_OUTPUT_KEYS = {
    'structured_ingest': {'manatuabon_schema', 'payload_type', 'summary', 'significance', 'domain_tags', 'source_catalogs'},
    'skeptic_review': {'verdict', 'reasoning', 'objections', 'score_contributions'},
    'archivist_review': {'verdict', 'reasoning', 'objections', 'score_contributions'},
    'judge_review': {'verdict', 'reasoning', 'scores', 'final_score', 'merge_target'},
    'reflection_review': {
        'verdict',
        'reasoning',
        'objections',
        'score_contributions',
        'recommended_decision',
        'rerun_worthy',
        'blockers',
        'concrete_revisions',
        'evidence_requests',
    },
    'evidence_request': {'request_text', 'priority', 'source_agent', 'source_context'},
}

ALLOWED_VERDICTS = {
    'skeptic_review': {'weak', 'plausible', 'strong'},
    'archivist_review': {'unique', 'partial_overlap', 'duplicate'},
    'judge_review': {'accepted', 'rejected', 'needs_revision', 'held', 'merged'},
    'reflection_review': {'revise_and_retry', 'hold_for_evidence', 'ready_for_rejudge', 'reject_later'},
}


def generate_json_response(sample, max_new_tokens=768):
    prompt = format_prompt(sample)
    encoded = tokenizer(prompt, return_tensors='pt').to(model.device)
    generated = model.generate(**encoded, max_new_tokens=max_new_tokens, use_cache=True)
    text = tokenizer.decode(generated[0], skip_special_tokens=True)
    response_text = text.split('Assistant JSON:\n', 1)[-1].strip()
    parsed = json.loads(response_text)
    return response_text, parsed


def validate_generated_output(task_type, parsed_output):
    missing = REQUIRED_OUTPUT_KEYS[task_type].difference(parsed_output.keys())
    if missing:
        return False, f'Missing keys: {sorted(missing)}'
    if task_type == 'structured_ingest':
        if parsed_output.get('manatuabon_schema') != 'structured_ingest_v1':
            return False, 'manatuabon_schema must equal structured_ingest_v1'
        significance = parsed_output.get('significance')
        if not isinstance(significance, (int, float)) or not 0.0 <= significance <= 1.0:
            return False, 'significance must be numeric in [0, 1]'
    if task_type in ALLOWED_VERDICTS:
        verdict = parsed_output.get('verdict')
        if verdict not in ALLOWED_VERDICTS[task_type]:
            return False, f'Unexpected verdict: {verdict!r}'
    if task_type == 'judge_review':
        final_score = parsed_output.get('final_score')
        if not isinstance(final_score, (int, float)) or not 0.0 <= final_score <= 1.0:
            return False, 'final_score must be numeric in [0, 1]'
    return True, 'ok'


sample_count = min(6, len(raw_eval_data))
results = []
for index in range(sample_count):
    sample = raw_eval_data[index]
    try:
        _, parsed_output = generate_json_response(sample)
        passed, detail = validate_generated_output(sample['task_type'], parsed_output)
    except Exception as exc:
        parsed_output = None
        passed = False
        detail = f'{type(exc).__name__}: {exc}'
    results.append({
        'record_id': sample['record_id'],
        'task_type': sample['task_type'],
        'passed': passed,
        'detail': detail,
        'expected_output': sample['output'],
        'parsed_output': parsed_output,
    })

passed_count = sum(item['passed'] for item in results)
print(f'Contract checks passed: {passed_count}/{len(results)}')
for item in results:
    print('-' * 80)
    print(item['record_id'], item['task_type'], item['passed'])
    print(item['detail'])
    if not item['passed']:
        pprint(item['expected_output'])
        pprint(item['parsed_output'])

In [ ]:
# 10. Optional: copy adapter to Google Drive
from importlib import import_module
import shutil

drive = import_module('google.colab.drive')
drive.mount('/content/drive')
shutil.copytree(OUTPUT_DIR, '/content/drive/MyDrive/manatuabon_nemotron_qlora', dirs_exist_ok=True)
print('Saved to Drive.')